# **LONG STEPS in GRADIENT DESCENT METHODS**

# Install

### Firedrake

In [ ]:
try:
    import google.colab  # noqa: F401
except ImportError:
    from firedrake import *
else:
    try:
        from firedrake import *
    except ImportError:
        !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh"
        !bash "/tmp/firedrake-install.sh"
        from firedrake import *

## **Motivation:** When Newton fails

We're all used to using Newton, and who can blame us?

When it works, the convergence is superb ($\mathcal{O}(C^{- n^2})$*).
However, it's not always the best tool for the job:

1. Newton–Kantorovich (i.e. the convergence result for Newton in the continuous setting) requires **local Lipschitz differentiability** of the residual. It's relatively easy to cook up situations where this is violated, e.g. say we want to minimise this funky functional:
$$
E(u) := \frac{1}{2}\int\|\nabla u\|^2 + \frac{\lambda}{2}\int_{u \le f}(u-f)^2.
$$
This kind of functional** is pretty much the simplest way to approach constrained optimisation, i.e. as $\lambda \to \infty$, we expect the minimiser of $E$ to be the $u$ that minimises $\int \|\nabla u\|^2$ subject to $u \ge f$.

    However, the (Fréchet) derivative of $E$,
    $$
    E'(u; v) = \int \nabla u \cdot \nabla v + \lambda\int_{u \le f}(u - f)v,
    $$
    is **not Lipschitz differentiable** in $H^1$.
    No surprise then, we see Newton fails to minimise $E$.

> **It's a bit technical about whether this result holds in the continuous setting, but let's just assume it does.*

> ***C.F. Nemystkii operators.*

In [ ]:
# Parameters
nx = 32
p = 1
gamma = Constant(2**8)
max_iters = 20

# Mesh
mesh = UnitSquareMesh(nx, nx, quadrilateral=True)
V = FunctionSpace(mesh, "Q", 1)

# Constraint function f(x, y): circular bump
x, y = SpatialCoordinate(mesh)
f_center = (0.5, 0.5)
f_width = 0.2
f = conditional(
    (x - f_center[0])**2 + (y - f_center[1])**2 <= f_width**2,
    1.0,
    0.0
)

# Functions
u = Function(V)
u_ = Function(V)
u_.interpolate(1)
v = TestFunction(V)

# BCs
bcs = DirichletBC(V, 0.0, "on_boundary")

# Energy
h1_norm = 0.5 * inner(grad(u_), grad(u_)) * dx
constraint_viol = 0.5 * conditional(gt(f - u_, 0), f - u_, 0)**2 * dx
E = h1_norm + gamma * constraint_viol
print(assemble(E))

# Residual
E_prime = derivative(E, u_, v)
F = E_prime + derivative(E_prime, u_, u - u_)

# Newton's method with live energy plot
# plt.ion()
# fig, ax = plt.subplots()
# energy_line, = ax.plot([], [], marker='o')
# ax.set_xlabel('Iteration')
# ax.set_ylabel('Energy')
# ax.set_title('Energy evolution (Newton)')
# energies = [assemble(E)]
print(assemble(E))
for k in range(max_iters):
    solve(F == 0, u, bcs=bcs)
    print(assemble(u**2 * dx))
    u_.assign(u)
    print(assemble(E))
    # energies += assemble(E)
    # print(f"Newton iter {k+1}: E={curr_E:.6e}")
    # # Update live plot
    # energy_line.set_data(range(len(energies)), energies)
    # ax.relim()
    # ax.autoscale_view()
    # clear_output(wait=True)
    # plt.draw()
    # plt.pause(0.1)
# plt.ioff()
# plt.show()

# # Plot result (if matplotlib available)
# try:
#     tripcolor(u)
#     plt.title("Newton minimiser u")
#     plt.show()
# except Exception:
#     pass

2. For nonlinear problems with many solutions, initial guesses may lie outside any basin of attraction.

3. At very high order, nonlinearity makes assembly costly.

Let's discuss a pretty robust alternative...

## **Back to basics:** Gradient descent (GD)

Let's restrict our attention to energy minimisation problems.
You might not have seen GD in the function space setting before, but the idea's identical.
Instead of the update
$$
\mathbf{x}_{n+1} = \mathbf{x}_n - h\nabla E(\mathbf{x}_n),
$$
we have
$$
(u_{n+1}, v) = (u_n, v) - hE'(\mathbf{x}_n; v).
$$

Key facts:
- Converges for fixed step sizes in (0, 2/L), with L the Lipschitz constant of \nabla E.
- Only requires one lower level of regularity than Newton.
- The linear operator to invert in function spaces is the inner product; often identical across steps.
- Global convergence to a minimizer for convex energies regardless of start.